In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image


class StressColorConverter:
    def __init__(
        self,
        input_dir=r"E:\RFNS\CODE-R2",
        output_dir=r"E:\RFNS\CODE-R2\converted_blue",
        background_threshold=245,
        alpha_threshold=5,
        saturation_min=0.12,
        value_min=0.08,
        low_color=(198, 226, 245),
        high_color=(8, 48, 107),
        gamma=1.0,
        transparent_background_to_white=True,
        make_colorbar=True,
        colorbar_width=70,
        colorbar_height=420,
        debug=True,
    ):
        # Adjustable parameters
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.background_threshold = background_threshold
        self.alpha_threshold = alpha_threshold
        self.saturation_min = saturation_min
        self.value_min = value_min
        self.low_color = np.array(low_color, dtype=np.float32)
        self.high_color = np.array(high_color, dtype=np.float32)
        self.gamma = gamma
        self.transparent_background_to_white = transparent_background_to_white
        self.make_colorbar = make_colorbar
        self.colorbar_width = colorbar_width
        self.colorbar_height = colorbar_height
        self.debug = debug

        self.output_dir.mkdir(parents=True, exist_ok=True)

    def run(self):
        files = sorted(self.input_dir.glob("*.png"))

        if not files:
            print(f"No PNG files found in: {self.input_dir}")
            return

        print(f"Found {len(files)} PNG files.")

        for file_path in files:
            self.convert_one(file_path)

        if self.make_colorbar:
            self.save_colorbar()

        print(f"Saved converted images to: {self.output_dir}")

    def convert_one(self, file_path):
        image = Image.open(file_path)
        original_mode = image.mode
        has_alpha = self._has_alpha(image)

        rgba = np.asarray(image.convert("RGBA")).astype(np.float32)

        rgb = rgba[..., :3]
        alpha = rgba[..., 3]

        hsv = self._rgb_to_hsv(rgb)

        h = hsv[..., 0]
        s = hsv[..., 1]
        v = hsv[..., 2]

        transparent_mask = alpha <= self.alpha_threshold
        white_mask = np.all(rgb >= self.background_threshold, axis=-1)
        background_mask = transparent_mask | white_mask

        # Convert only saturated contour pixels. Preserve gray/black boundary lines.
        color_mask = (
            (~background_mask)
            & (s >= self.saturation_min)
            & (v >= self.value_min)
        )

        level = self._abaqus_hue_to_level(h)
        level = np.clip(level, 0.0, 1.0)
        level = level ** self.gamma

        converted_rgb = rgb.copy()
        blue_rgb = self._level_to_blue(level)

        converted_rgb[color_mask] = blue_rgb[color_mask]

        if self.transparent_background_to_white:
            output_rgba = self._make_white_background(converted_rgb, alpha, background_mask)
        else:
            output_rgba = self._preserve_alpha_background(converted_rgb, alpha, white_mask, transparent_mask)

        output = np.clip(output_rgba, 0, 255).astype(np.uint8)
        output_image = Image.fromarray(output, mode="RGBA")

        output_path = self.output_dir / f"{file_path.stem}_blue.png"
        output_image.save(output_path)

        if self.debug:
            self._print_image_info(
                file_path=file_path,
                original_mode=original_mode,
                size=image.size,
                has_alpha=has_alpha,
                transparent_mask=transparent_mask,
                white_mask=white_mask,
                background_mask=background_mask,
                color_mask=color_mask,
                s=s,
                h=h,
                output_path=output_path,
            )

    def save_colorbar(self):
        border = 2
        w = self.colorbar_width
        h = self.colorbar_height

        canvas = np.ones((h + 2 * border, w + 2 * border, 4), dtype=np.float32) * 255.0
        canvas[..., 3] = 255.0

        levels = np.linspace(1.0, 0.0, h, dtype=np.float32)[:, None]
        levels = np.repeat(levels, w, axis=1)

        colors = self._level_to_blue(levels)
        alpha = np.ones((h, w, 1), dtype=np.float32) * 255.0
        colorbar = np.concatenate([colors, alpha], axis=-1)

        canvas[border:border + h, border:border + w, :] = colorbar

        # Border
        canvas[:border, :, :3] = 0
        canvas[-border:, :, :3] = 0
        canvas[:, :border, :3] = 0
        canvas[:, -border:, :3] = 0

        output = np.clip(canvas, 0, 255).astype(np.uint8)
        output_image = Image.fromarray(output, mode="RGBA")

        output_path = self.output_dir / "blue_colorbar.png"
        output_image.save(output_path)

        if self.debug:
            print(f"Colorbar saved: {output_path.name}")

    def _make_white_background(self, converted_rgb, alpha, background_mask):
        alpha_norm = alpha[..., None] / 255.0

        white = np.ones_like(converted_rgb) * 255.0
        composite_rgb = converted_rgb * alpha_norm + white * (1.0 - alpha_norm)

        composite_rgb[background_mask] = 255.0

        output_alpha = np.ones(alpha.shape, dtype=np.float32) * 255.0
        output_rgba = np.concatenate([composite_rgb, output_alpha[..., None]], axis=-1)

        return output_rgba

    def _preserve_alpha_background(self, converted_rgb, alpha, white_mask, transparent_mask):
        converted_rgb[white_mask] = 255.0
        converted_rgb[transparent_mask] = 255.0

        output_rgba = np.concatenate([converted_rgb, alpha[..., None]], axis=-1)

        return output_rgba

    def _print_image_info(
        self,
        file_path,
        original_mode,
        size,
        has_alpha,
        transparent_mask,
        white_mask,
        background_mask,
        color_mask,
        s,
        h,
        output_path,
    ):
        total_pixels = color_mask.size
        transparent_pixels = int(np.sum(transparent_mask))
        white_pixels = int(np.sum(white_mask & (~transparent_mask)))
        background_pixels = int(np.sum(background_mask))
        converted_pixels = int(np.sum(color_mask))
        kept_non_bg_pixels = int(np.sum((~background_mask) & (~color_mask)))

        transparent_ratio = transparent_pixels / total_pixels * 100.0
        white_ratio = white_pixels / total_pixels * 100.0
        background_ratio = background_pixels / total_pixels * 100.0
        converted_ratio = converted_pixels / total_pixels * 100.0

        if converted_pixels > 0:
            hue_used = h[color_mask]
            sat_used = s[color_mask]
            hue_min = float(np.percentile(hue_used, 1))
            hue_max = float(np.percentile(hue_used, 99))
            sat_mean = float(np.mean(sat_used))
            hue_text = f"hue_p1_p99=({hue_min:.1f}, {hue_max:.1f}), sat_mean={sat_mean:.3f}"
        else:
            hue_text = "hue_p1_p99=N/A, sat_mean=N/A"

        print(
            f"{file_path.name} | "
            f"size={size}, mode={original_mode}, alpha={has_alpha}, "
            f"transparent={transparent_pixels}({transparent_ratio:.2f}%), "
            f"white_bg={white_pixels}({white_ratio:.2f}%), "
            f"all_bg={background_pixels}({background_ratio:.2f}%), "
            f"converted={converted_pixels}({converted_ratio:.2f}%), "
            f"kept_non_bg={kept_non_bg_pixels}, "
            f"{hue_text} -> {output_path.name}"
        )

    def _has_alpha(self, image):
        if image.mode in ("RGBA", "LA"):
            return True
        if image.mode == "P" and "transparency" in image.info:
            return True
        return False

    def _abaqus_hue_to_level(self, hue):
        """
        Abaqus rainbow order:
        blue/cyan/green/yellow/red = low to high.
        Return normalized level: 0 low, 1 high.
        """
        level = np.zeros_like(hue, dtype=np.float32)

        main_mask = hue <= 240.0
        level[main_mask] = (240.0 - hue[main_mask]) / 240.0

        red_wrap_mask = hue >= 300.0
        level[red_wrap_mask] = 1.0

        purple_mask = (hue > 240.0) & (hue < 300.0)
        level[purple_mask] = 0.0

        return level

    def _level_to_blue(self, level):
        """
        level = 0: light blue
        level = 1: dark blue
        """
        level_3d = level[..., None]
        return self.low_color * (1.0 - level_3d) + self.high_color * level_3d

    def _rgb_to_hsv(self, rgb):
        rgb_norm = rgb / 255.0

        r = rgb_norm[..., 0]
        g = rgb_norm[..., 1]
        b = rgb_norm[..., 2]

        maxc = np.max(rgb_norm, axis=-1)
        minc = np.min(rgb_norm, axis=-1)
        delta = maxc - minc

        h = np.zeros_like(maxc, dtype=np.float32)

        r_mask = (maxc == r) & (delta != 0)
        g_mask = (maxc == g) & (delta != 0)
        b_mask = (maxc == b) & (delta != 0)

        h[r_mask] = 60.0 * (((g[r_mask] - b[r_mask]) / delta[r_mask]) % 6.0)
        h[g_mask] = 60.0 * (((b[g_mask] - r[g_mask]) / delta[g_mask]) + 2.0)
        h[b_mask] = 60.0 * (((r[b_mask] - g[b_mask]) / delta[b_mask]) + 4.0)

        s = np.zeros_like(maxc, dtype=np.float32)
        nonzero_mask = maxc != 0
        s[nonzero_mask] = delta[nonzero_mask] / maxc[nonzero_mask]

        v = maxc

        hsv = np.stack([h, s, v], axis=-1)
        return hsv


converter = StressColorConverter(
    input_dir=r"E:\RFNS\CODE-R2",
    output_dir=r"E:\RFNS\CODE-R2\converted_blue",
    background_threshold=245,
    alpha_threshold=5,
    saturation_min=0.12,
    value_min=0.08,
    low_color=(198, 226, 245),
    high_color=(8, 48, 107),
    gamma=1.0,
    transparent_background_to_white=True,
    make_colorbar=True,
    colorbar_width=70,
    colorbar_height=420,
    debug=True,
)

converter.run()

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize


class StressPerceptualColorConverter:
    def __init__(
        self,
        input_dir=r"E:\RFNS\CODE-R2",
        output_dir=r"E:\RFNS\CODE-R2\converted_cividis_transparent",
        target_cmap="cividis",
        background_threshold=245,
        alpha_threshold=5,
        saturation_min=0.12,
        value_min=0.08,
        gamma=1.0,
        transparent_background=True,
        keep_gray_black_lines=True,
        make_normalized_colorbar=True,
        make_mpa_colorbars=True,
        colorbar_label="S, Mises (MPa)",
        colorbar_width_inch=1.05,
        colorbar_height_inch=4.2,
        colorbar_dpi=300,
        debug=True,
    ):
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.target_cmap = target_cmap

        self.background_threshold = background_threshold
        self.alpha_threshold = alpha_threshold
        self.saturation_min = saturation_min
        self.value_min = value_min
        self.gamma = gamma

        self.transparent_background = transparent_background
        self.keep_gray_black_lines = keep_gray_black_lines

        self.make_normalized_colorbar = make_normalized_colorbar
        self.make_mpa_colorbars = make_mpa_colorbars
        self.colorbar_label = colorbar_label
        self.colorbar_width_inch = colorbar_width_inch
        self.colorbar_height_inch = colorbar_height_inch
        self.colorbar_dpi = colorbar_dpi
        self.debug = debug

        self.output_dir.mkdir(parents=True, exist_ok=True)

        # MPa ranges manually read from your Abaqus legends.
        # If you add more images later, add them here.
        self.stress_ranges_mpa = {
            "1.png": (0.0, 37.22),
            "2.png": (0.0, 38.53),
            "3.png": (0.0, 29.18),
            "4.png": (0.0, 77.84),
            "5.png": (0.0, 76.77),
        }

        self.cmap = cm.get_cmap(self.target_cmap)

    def run(self):
        files = sorted(self.input_dir.glob("*.png"))

        if not files:
            print(f"No PNG files found in: {self.input_dir}")
            return

        print("=" * 90)
        print("Stress Perceptual Color Converter")
        print("=" * 90)
        print(f"Input directory : {self.input_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"Target colormap : {self.target_cmap}")
        print(f"Transparent bg  : {self.transparent_background}")
        print(f"Found PNG files : {len(files)}")
        print("=" * 90)

        for file_path in files:
            self.convert_one(file_path)

        if self.make_normalized_colorbar:
            self.save_normalized_colorbar()

        if self.make_mpa_colorbars:
            self.save_all_mpa_colorbars(files)

        print("=" * 90)
        print(f"Saved converted images and colorbars to: {self.output_dir}")
        print("=" * 90)

    def convert_one(self, file_path):
        image = Image.open(file_path)
        original_mode = image.mode
        has_alpha = self._has_alpha(image)

        rgba = np.asarray(image.convert("RGBA")).astype(np.float32)
        rgb = rgba[..., :3]
        alpha = rgba[..., 3]

        hsv = self._rgb_to_hsv(rgb)

        h = hsv[..., 0]
        s = hsv[..., 1]
        v = hsv[..., 2]

        transparent_mask = alpha <= self.alpha_threshold
        white_mask = np.all(rgb >= self.background_threshold, axis=-1)
        background_mask = transparent_mask | white_mask

        # Saturated colored pixels are treated as Abaqus rainbow contour pixels.
        color_mask = (
            (~background_mask)
            & (s >= self.saturation_min)
            & (v >= self.value_min)
        )

        # Gray or black non-background pixels, such as boundary lines, are preserved.
        line_mask = (~background_mask) & (~color_mask)

        level = self._abaqus_hue_to_level(h)
        level = np.clip(level, 0.0, 1.0)
        level = level ** self.gamma

        converted_rgb = rgb.copy()
        new_rgb = self._level_to_colormap_rgb(level)
        converted_rgb[color_mask] = new_rgb[color_mask]

        output_rgba = self._compose_output_rgba(
            converted_rgb=converted_rgb,
            alpha=alpha,
            background_mask=background_mask,
            color_mask=color_mask,
            line_mask=line_mask,
        )

        output = np.clip(output_rgba, 0, 255).astype(np.uint8)
        output_image = Image.fromarray(output, mode="RGBA")

        output_path = self.output_dir / f"{file_path.stem}_{self.target_cmap}_transparent.png"
        output_image.save(output_path)

        if self.debug:
            self._print_image_info(
                file_path=file_path,
                original_mode=original_mode,
                size=image.size,
                has_alpha=has_alpha,
                transparent_mask=transparent_mask,
                white_mask=white_mask,
                background_mask=background_mask,
                color_mask=color_mask,
                line_mask=line_mask,
                s=s,
                h=h,
                output_path=output_path,
            )

    def _compose_output_rgba(self, converted_rgb, alpha, background_mask, color_mask, line_mask):
        h, w = alpha.shape
        output_rgba = np.zeros((h, w, 4), dtype=np.float32)

        output_rgba[..., :3] = converted_rgb
        output_rgba[..., 3] = alpha

        if self.transparent_background:
            # Make background truly transparent.
            output_rgba[background_mask, :3] = 255.0
            output_rgba[background_mask, 3] = 0.0
        else:
            # White background.
            output_rgba[background_mask, :3] = 255.0
            output_rgba[background_mask, 3] = 255.0

        # Ensure colored structure is visible.
        output_rgba[color_mask, 3] = np.maximum(output_rgba[color_mask, 3], 255.0)

        if self.keep_gray_black_lines:
            output_rgba[line_mask, 3] = np.maximum(output_rgba[line_mask, 3], 255.0)
        else:
            # If you do not want black/gray lines, make them transparent too.
            output_rgba[line_mask, 3] = 0.0

        return output_rgba

    def save_normalized_colorbar(self):
        output_path = self.output_dir / f"{self.target_cmap}_colorbar_normalized.png"

        fig, ax = plt.subplots(
            figsize=(self.colorbar_width_inch, self.colorbar_height_inch),
            dpi=self.colorbar_dpi,
        )

        norm = Normalize(vmin=0.0, vmax=1.0)
        sm = cm.ScalarMappable(norm=norm, cmap=self.cmap)
        sm.set_array([])

        cbar = fig.colorbar(sm, cax=ax)
        cbar.set_label("Normalized value", fontsize=8)
        cbar.ax.tick_params(labelsize=7, width=0.6, length=2.5)

        for spine in cbar.ax.spines.values():
            spine.set_linewidth(0.6)

        fig.savefig(
            output_path,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.04,
        )
        plt.close(fig)

        if self.debug:
            print(f"Colorbar saved: {output_path.name}")

    def save_all_mpa_colorbars(self, files):
        for file_path in files:
            key = file_path.name

            if key not in self.stress_ranges_mpa:
                if self.debug:
                    print(f"No MPa range found for {key}; skipped MPa colorbar.")
                continue

            vmin, vmax = self.stress_ranges_mpa[key]
            output_path = self.output_dir / f"{file_path.stem}_{self.target_cmap}_colorbar_MPa.png"
            self.save_one_mpa_colorbar(vmin, vmax, output_path)

            if self.debug:
                print(f"MPa colorbar saved: {output_path.name}, range=({vmin}, {vmax}) MPa")

    def save_one_mpa_colorbar(self, vmin, vmax, output_path):
        fig, ax = plt.subplots(
            figsize=(self.colorbar_width_inch, self.colorbar_height_inch),
            dpi=self.colorbar_dpi,
        )

        norm = Normalize(vmin=vmin, vmax=vmax)
        sm = cm.ScalarMappable(norm=norm, cmap=self.cmap)
        sm.set_array([])

        cbar = fig.colorbar(sm, cax=ax)
        cbar.set_label(self.colorbar_label, fontsize=8)
        cbar.ax.tick_params(labelsize=7, width=0.6, length=2.5)

        for spine in cbar.ax.spines.values():
            spine.set_linewidth(0.6)

        fig.savefig(
            output_path,
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.04,
        )
        plt.close(fig)

    def _level_to_colormap_rgb(self, level):
        rgba = self.cmap(level)
        rgb = rgba[..., :3] * 255.0
        return rgb.astype(np.float32)

    def _abaqus_hue_to_level(self, hue):
        """
        Approximate mapping from Abaqus rainbow hue to normalized scalar level.

        Abaqus-like rainbow order:
        blue/cyan/green/yellow/red = low to high.

        Return:
        level = 0 means low value.
        level = 1 means high value.
        """
        level = np.zeros_like(hue, dtype=np.float32)

        main_mask = hue <= 240.0
        level[main_mask] = (240.0 - hue[main_mask]) / 240.0

        red_wrap_mask = hue >= 300.0
        level[red_wrap_mask] = 1.0

        purple_mask = (hue > 240.0) & (hue < 300.0)
        level[purple_mask] = 0.0

        return level

    def _print_image_info(
        self,
        file_path,
        original_mode,
        size,
        has_alpha,
        transparent_mask,
        white_mask,
        background_mask,
        color_mask,
        line_mask,
        s,
        h,
        output_path,
    ):
        total_pixels = color_mask.size

        transparent_pixels = int(np.sum(transparent_mask))
        white_pixels = int(np.sum(white_mask & (~transparent_mask)))
        background_pixels = int(np.sum(background_mask))
        converted_pixels = int(np.sum(color_mask))
        line_pixels = int(np.sum(line_mask))

        transparent_ratio = transparent_pixels / total_pixels * 100.0
        white_ratio = white_pixels / total_pixels * 100.0
        background_ratio = background_pixels / total_pixels * 100.0
        converted_ratio = converted_pixels / total_pixels * 100.0
        line_ratio = line_pixels / total_pixels * 100.0

        if converted_pixels > 0:
            hue_used = h[color_mask]
            sat_used = s[color_mask]
            hue_min = float(np.percentile(hue_used, 1))
            hue_max = float(np.percentile(hue_used, 99))
            sat_mean = float(np.mean(sat_used))
            hue_text = f"hue_p1_p99=({hue_min:.1f}, {hue_max:.1f}), sat_mean={sat_mean:.3f}"
        else:
            hue_text = "hue_p1_p99=N/A, sat_mean=N/A"

        if file_path.name in self.stress_ranges_mpa:
            vmin, vmax = self.stress_ranges_mpa[file_path.name]
            range_text = f"MPa_range=({vmin:.3g}, {vmax:.3g})"
        else:
            range_text = "MPa_range=not provided"

        print(
            f"{file_path.name} | "
            f"size={size}, mode={original_mode}, alpha={has_alpha}, "
            f"transparent={transparent_pixels}({transparent_ratio:.2f}%), "
            f"white_bg={white_pixels}({white_ratio:.2f}%), "
            f"all_bg={background_pixels}({background_ratio:.2f}%), "
            f"converted={converted_pixels}({converted_ratio:.2f}%), "
            f"lines_kept={line_pixels}({line_ratio:.2f}%), "
            f"{hue_text}, "
            f"{range_text} -> {output_path.name}"
        )

    def _has_alpha(self, image):
        if image.mode in ("RGBA", "LA"):
            return True
        if image.mode == "P" and "transparency" in image.info:
            return True
        return False

    def _rgb_to_hsv(self, rgb):
        rgb_norm = rgb / 255.0

        r = rgb_norm[..., 0]
        g = rgb_norm[..., 1]
        b = rgb_norm[..., 2]

        maxc = np.max(rgb_norm, axis=-1)
        minc = np.min(rgb_norm, axis=-1)
        delta = maxc - minc

        h = np.zeros_like(maxc, dtype=np.float32)

        r_mask = (maxc == r) & (delta != 0)
        g_mask = (maxc == g) & (delta != 0)
        b_mask = (maxc == b) & (delta != 0)

        h[r_mask] = 60.0 * (((g[r_mask] - b[r_mask]) / delta[r_mask]) % 6.0)
        h[g_mask] = 60.0 * (((b[g_mask] - r[g_mask]) / delta[g_mask]) + 2.0)
        h[b_mask] = 60.0 * (((r[b_mask] - g[b_mask]) / delta[b_mask]) + 4.0)

        s = np.zeros_like(maxc, dtype=np.float32)
        nonzero_mask = maxc != 0
        s[nonzero_mask] = delta[nonzero_mask] / maxc[nonzero_mask]

        v = maxc

        hsv = np.stack([h, s, v], axis=-1)
        return hsv


converter = StressPerceptualColorConverter(
    input_dir=r"E:\RFNS\CODE-R2",
    output_dir=r"E:\RFNS\CODE-R2\converted_cividis_transparent",
    target_cmap="viridis",

    background_threshold=245,
    alpha_threshold=5,
    saturation_min=0.12,
    value_min=0.08,

    gamma=1.0,

    transparent_background=True,
    keep_gray_black_lines=True,

    make_normalized_colorbar=True,
    make_mpa_colorbars=True,
    colorbar_label="S, Mises (MPa)",

    colorbar_width_inch=1.05,
    colorbar_height_inch=4.2,
    colorbar_dpi=300,

    debug=True,
)

converter.run()